In [3]:
import pandas as p

In [4]:
genes = {g.orf.lower(): g for g in Gene.objects.all()}

In [5]:
strains = {}
for s in Strain.objects.select_related():
    strains.setdefault(s.label().lower(), []).append(s)

In [25]:
df = p.read_table('/mnt/storage/151111/living/network_2016-10-21/layout_coordinates.txt', sep=';')

In [6]:
ds = Dataset.objects.get(pk=19)

In [5]:
corrs = {q.label().lower(): q for q in ds.correlation_axis.select_related()}

In [7]:
for l in df.label:
    if l not in corrs:
        print(l)

arp3-d11a
cdc14-3
eaf1
est2
flo5
hmlalpha1
ho
las17-1
pfa3
pob3-q308k
stt3-1
ylr161w


In [18]:
import json

In [11]:
nodes = json.load(open('/home/matej/dev/workspace/thecellmap/static/visualization/Living/nodes.json'))

In [13]:
node_map = {n['label'].lower(): n['id'] for n in nodes['nodes']}

In [26]:
res = []
for l, x, y, _ in df.itertuples(index=False):
    if l not in node_map:
        continue
    
    res.append({'id': node_map[l], 'x':x, 'y':y})

In [27]:
newlayout = json.dumps({'nodes':res}).replace(' ', '')

In [28]:
with open('/home/matej/dev/workspace/thecellmap/static/visualization/Living/layout.json', 'w') as out:
    out.write(newlayout)

In [29]:
foo=set()
for l in df.label:
    if l in foo: print(l)
    foo.add(l)

In [28]:
# df = p.read_excel('/mnt/storage/151111/living/network_2016-10-21/SAFE analysis - live dataset_201610221.xls', sheetname=1)
df = p.read_table('/mnt/storage/151111/living/network_2016-10-21/node_properties_annotation-highest.txt', skip_blank_lines=True, skiprows=3)

In [43]:
terms = {}
for l in open('/mnt/storage/151111/living/network_2016-10-21/domain_properties_annotation-highest.txt').readlines()[5:]:
    i, name, r, g, b = l.strip().split('\t')
    color = '%02x%02x%02x' % tuple(map(int, (r,g,b)))
    term = Term.objects.create(
        annotation_id=17,
        name=name,
        alias=name,
        source='Wen Wang',
        color=color
    )
    terms[int(i)] = term

In [54]:
df = df[(df.iloc[:,4] > 0) & (df.iloc[:,2] > 1)]
for lbl, orf, domain in df.iloc[:,:3].itertuples(index=False):
    term = terms[domain]
    
    strain = strains[lbl]
    gene = genes[orf.lower()]
    
    print strain, gene
    
    term.genes.add(gene)
    for s in strain:
        term.strains.add(s)

[<Strain: YBL074C (AAR2) - aar2-5001 - tsa1108>, <Strain: YBL074C (AAR2) - aar2-5001 - tsq1352>] YBL074C (AAR2)
[<Strain: YKL112W (ABF1) - abf1-103 - tsq1268>] YKL112W (ABF1)
[<Strain: YMR072W (ABF2) - sn1446>] YMR072W (ABF2)
[<Strain: YNR016C (ACC1) - acc1-5007-supp1 - tsq2274>] YNR016C (ACC1)
[<Strain: YNR016C (ACC1) - acc1-5009-supp1 - tsq2276>] YNR016C (ACC1)
[<Strain: YDL203C (ACK1) - dma801>, <Strain: YDL203C (ACK1) - sn2778>] YDL203C (ACK1)
[<Strain: YFL039C (ACT1) - act1-101 - tsa136>, <Strain: YFL039C (ACT1) - act1-101 - tsq136>] YFL039C (ACT1)
[<Strain: YFL039C (ACT1) - act1-105 - tsq1209>, <Strain: YFL039C (ACT1) - act1-105 - tsa909>] YFL039C (ACT1)
[<Strain: YFL039C (ACT1) - act1-108 - tsa904>, <Strain: YFL039C (ACT1) - act1-108 - tsq1206>] YFL039C (ACT1)
[<Strain: YFL039C (ACT1) - act1-111 - tsa214>] YFL039C (ACT1)
[<Strain: YFL039C (ACT1) - act1-112 - tsq218>] YFL039C (ACT1)
[<Strain: YFL039C (ACT1) - act1-119 - tsa215>, <Strain: YFL039C (ACT1) - act1-119 - tsq1093>] YFL0

In [8]:
colours = [c for c, in Term.objects.filter(annotation=15).values_list('color')]

In [38]:
import random
random.shuffle(colours)

In [41]:
for i, d in enumerate(df['domain name'].unique()):
    Term.objects.create(
        annotation_id=17,
        name=d,
        alias=d,
        source='Wen Wang',
        color=colours[i],
    )

In [9]:
terms = {t.name: t for t in Term.objects.filter(annotation=17)}

In [14]:
for _, term, orf, gene, allele in df.itertuples(index=False):
    term = terms[term]
    if allele not in strains:
        print(allele)
    if orf.lower() not in genes and allele not in genes:
        print(orf)
    
    strain = strains[allele]
    if orf.lower() in genes:
        gene = genes[orf.lower()]
    else:
        gene = genes[allele]
    
    print strain, gene
    
    term.genes.add(gene)
    for s in strain:
        term.strains.add(s)

[<Strain: YAL023C (PMT2) - dma31>, <Strain: YAL023C (PMT2) - sn870>] YAL023C (PMT2)
[<Strain: YAL026C (DRS2) - drs2-supp1 - dma30>] YAL026C (DRS2)
[<Strain: YAL041W (CDC24) - cdc24-1 - tsa408>, <Strain: YAL041W (CDC24) - cdc24-1 - tsq408>] YAL041W (CDC24)
[<Strain: YAL041W (CDC24) - cdc24-11 - tsa148>, <Strain: YAL041W (CDC24) - cdc24-11 - tsq148>] YAL041W (CDC24)
[<Strain: YAL041W (CDC24) - cdc24-5 - tsa304>] YAL041W (CDC24)
[<Strain: YAL041W (CDC24) - cdc24-h - tsa42>, <Strain: YAL041W (CDC24) - cdc24-h - tsq1065>] YAL041W (CDC24)
[<Strain: YAL056C-A - dma57>] YAL056C-A
[<Strain: YAL058W (CNE1) - dma56>, <Strain: YAL058W (CNE1) - sn2203>] YAL058W (CNE1)
[<Strain: YBL007C (SLA1) - dma91>, <Strain: YBL007C (SLA1) - sn761>] YBL007C (SLA1)
[<Strain: YBL020W (RFT1) - rft1-5028 - tsq2552>] YBL020W (RFT1)
[<Strain: YBL040C (ERD2) - erd2-5001 - tsa1072>] YBL040C (ERD2)
[<Strain: YBL047C (EDE1) - dma126>, <Strain: YBL047C (EDE1) - sn164>] YBL047C (EDE1)
[<Strain: YBL061C (SKT5) - dma137>, <St

In [47]:
strains

{u'fig4': [<Strain: YNL325C (FIG4) - sn780>],
 u'fig2': [<Strain: YCR089W (FIG2) - dma589>,
  <Strain: YCR089W (FIG2) - sn781>],
 u'fig1': [<Strain: YBR040W (FIG1) - dma210>,
  <Strain: YBR040W (FIG1) - sn782>],
 u'vps53': [<Strain: YJL029C (VPS53) - sn248>],
 u'vps52': [<Strain: YDR484W (VPS52) - sn249>],
 u'vps51': [<Strain: YKR020W (VPS51) - dma2932>,
  <Strain: YKR020W (VPS51) - sn420>],
 u'vps55': [<Strain: YJR044C (VPS55) - dma5320>,
  <Strain: YJR044C (VPS55) - sn271>],
 u'tcb1': [<Strain: YOR086C (TCB1) - dma4523>,
  <Strain: YOR086C (TCB1) - sn2500>],
 u'tcb3': [<Strain: YML072C (TCB3) - sn2501>,
  <Strain: YML072C (TCB3) - dma3582>],
 u'tcb2': [<Strain: YNL087W (TCB2) - dma4005>,
  <Strain: YNL087W (TCB2) - sn605>],
 u'tsc10-1': [<Strain: YBR265W (TSC10) - tsc10-1 - tsa543>,
  <Strain: YBR265W (TSC10) - tsc10-1 - tsq543>],
 u'ycr022c': [<Strain: YCR022C - dma544>],
 u'ykr011c': [<Strain: YKR011C - dma2918>, <Strain: YKR011C - sn3730>],
 u'clg1': [<Strain: YGL215W (CLG1) - dma

In [17]:
Term.objects.filter(annotation=17).delete()

(6522,
 {u'base.Term': 17, u'base.Term_genes': 2243, u'base.Term_strains': 4262})